In [1]:
import tensorflow as tf
import numpy as np
import os
import sys

project_root = os.path.abspath(os.path.join(".."))  # One level up from current script
if project_root not in sys.path:
    sys.path.append(project_root)

from keras.src.metrics.accuracy_metrics import accuracy
from src.model_loader import ModelLoader
from src.data_loader import DataLoader
from src.utils import get_class_weigths, get_confusion_matrix, get_classification_report

print(tf.__version__)

2.16.2


In [6]:
def is_valid_image(path):
    try:
        img_bytes = tf.io.read_file(path)
        decoded_img = tf.io.decode_image(img_bytes)
        return True
    except tf.errors.InvalidArgumentError as e:
        print(f"Found bad path {path}...{e}")
        return False

def clean_invalid_images(datasets_base_path):
    for root, dirs, files in os.walk(datasets_base_path):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                image_path = os.path.join(root, file)
                if not is_valid_image(image_path):
                    print(f"Removing invalid image: {image_path}")
                    os.remove(image_path)

clean_invalid_images("../datasets")

2025-04-11 10:13:27.731241: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile


Found bad path ../datasets/Painting/painting_02662.jpg...{{function_node __wrapped__DecodeImage_device_/job:localhost/replica:0/task:0/device:CPU:0}} Input size should match (header_size + row_size * abs_height) but they differ by 2 [Op:DecodeImage] name: 
Removing invalid image: ../datasets/Painting/painting_02662.jpg


2025-04-11 10:14:16.158706: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: INVALID_ARGUMENT: Input size should match (header_size + row_size * abs_height) but they differ by 2
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


In [4]:
# Préparation des datasets
data_loader = DataLoader()

datasets = {
    "binary_nocw": data_loader.load_binary_dataset(
        positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"]
    ),
    "binary_cw": data_loader.load_binary_dataset(
        positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"], class_weights=True
    ),
    "multiclass_nocw": data_loader.load_multiclass_dataset(
        class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"]
    ),
    "multiclass_cw": data_loader.load_multiclass_dataset(
        class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"], class_weights=True
    ),
}

KeyboardInterrupt: 

# CNN_HARD

In [3]:
cnn_hard_loader = ModelLoader(model_name="CNN_HARD")

history_cnn_all_ds = []
all_model_cnn = {}

with tf.device("/gpu:0"):
    for dataset_name, (train_data, val_data, test_data) in datasets.items():
        cnn_hard_model = cnn_hard_loader.create_model_CNN_hard()
        all_model_cnn[f"{dataset_name}"] = cnn_hard_model

        if "nocw" in dataset_name:
            history_cnn = cnn_hard_model.fit(
                train_data,
                validation_data=val_data,
                epochs=10,
                verbose=2,
                callbacks=[cnn_hard_loader.get_tensorboard_callback(), cnn_hard_loader.get_early_stopping(), cnn_hard_loader.get_model_checkpoint()],
            )
            history_cnn_all_ds.append(history_cnn)
        else:
            history_cnn = cnn_hard_model.fit(
                train_data,
                validation_data=val_data,
                epochs=10,
                verbose=2,
                class_weight=get_class_weigths(train_data, val_data),
                callbacks=[cnn_hard_loader.get_tensorboard_callback(), cnn_hard_loader.get_early_stopping(), cnn_hard_loader.get_model_checkpoint()],
            )
            history_cnn_all_ds.append(history_cnn)

/Users/tanguydumontier/PycharmProjects/CESI_DS/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

ValueError: When using `save_weights_only=True` in `ModelCheckpoint`, the filepath provided must end in `.weights.h5` (Keras weights format). Received: filepath=./../models/weights/CNN_HARD/20250411-100344_.h5

# RES_NET

In [ ]:
res_net_loader = ModelLoader(model_name="RES_NET")

history_resnet_all_ds = []
all_model_resnet = {}

for dataset_name, (train_data, val_data, test_data) in datasets.items():
    res_net_model = res_net_loader.create_model_CNN_hard()
    all_model_resnet[f"{dataset_name}"] = res_net_model

    if "nocw" in dataset_name:
        history_resnet = res_net_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            callbacks=[res_net_loader.get_tensorboard_callback(), res_net_loader.get_early_stopping(), res_net_loader.get_model_checkpoint()],
        )
        history_resnet_all_ds.append(history_resnet)
    else:
        history_resnet = res_net_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            class_weight=get_class_weigths(train_data, val_data),
            callbacks=[res_net_loader.get_tensorboard_callback(), res_net_loader.get_early_stopping(), res_net_loader.get_model_checkpoint()],
        )
        history_resnet_all_ds.append(history_resnet)

# INCEPTION

In [ ]:

inception_loader = ModelLoader(model_name="INCEPTION")

history_inception_all_ds = []
all_model_inception = {}

for dataset_name, (train_data, val_data, test_data) in datasets.items():
    inception_model = inception_loader.create_model_with_inception()
    all_model_inception[f"{dataset_name}"] = inception_model

    if "nocw" in dataset_name:
        history_inception = inception_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            callbacks=[inception_model.get_tensorboard_callback(), inception_model.get_early_stopping(), inception_model.get_model_checkpoint()],
        )
        history_inception_all_ds.append(history_inception)
    else:
        history_inception = inception_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=2,
            class_weight=get_class_weigths(train_data, val_data),
            callbacks=[inception_loader.get_tensorboard_callback(), inception_loader.get_early_stopping(), inception_loader.get_model_checkpoint()],
        )
        history_inception_all_ds.append(history_inception)